# ROC、PR 与 AUC

**面试回答：**ROC 观察 TPR/FPR，PR 观察 precision/recall；正类稀少的欺诈队列更应看 PR、recall@预算和实际审核量。

## 真实案例

10 笔订单只有 3 笔欺诈，模型输出风险分数，审核团队每天最多处理 3 笔。

In [1]:
import numpy as np  # 导入 NumPy 手写指标。
order=np.array(['F01','F02','F03','F04','F05','F06','F07','F08','F09','F10'])  # 构造订单编号。
y=np.array([0,0,1,0,0,1,0,0,1,0])  # 标记真实欺诈。
score=np.array([.05,.16,.82,.30,.22,.67,.12,.41,.55,.08])  # 记录模型风险得分。
print('订单 | 欺诈 | 风险分')  # 输出账本表头。
for a,b,c in zip(order,y,score):  # 逐条展示样本。
    print(a,b,c)  # 输出订单。

订单 | 欺诈 | 风险分
F01 0 0.05
F02 0 0.16
F03 1 0.82
F04 0 0.3
F05 0 0.22
F06 1 0.67
F07 0 0.12
F08 0 0.41
F09 1 0.55
F10 0 0.08


## Baseline / 基线

基线随机排序，期望 precision 约等于欺诈先验。

In [2]:
prevalence=float(y.mean())  # 计算正类先验。
print('随机排序期望 precision=',round(prevalence,3))  # 输出随机基线。
print('全预测正常 accuracy=',round(float(np.mean(y==0)),3))  # 展示 accuracy 在不平衡下的误导性。

随机排序期望 precision= 0.3
全预测正常 accuracy= 0.7


In [3]:
def metrics(threshold):  # 定义阈值下的分类指标。
    pred=score>=threshold  # 根据阈值生成审核决策。
    tp=int(np.sum(pred&(y==1)))  # 统计真阳性。
    fp=int(np.sum(pred&(y==0)))  # 统计假阳性。
    fn=int(np.sum((~pred)&(y==1)))  # 统计假阴性。
    tpr=tp/(tp+fn) if tp+fn else 0.0  # 计算召回率。
    fpr=fp/np.sum(y==0)  # 计算假阳性率。
    precision=tp/(tp+fp) if tp+fp else 1.0  # 计算精确率。
    return tpr,fpr,precision,int(pred.sum())  # 返回 ROC、PR 与审核量。
rows=[(t,)+metrics(t) for t in sorted(set(score),reverse=True)]  # 枚举所有得分阈值。
print('阈值 | TPR | FPR | Precision | 审核量')  # 输出曲线离散点表头。
for row in rows:  # 逐点输出指标。
    print(round(row[0],2),*[round(v,3) if isinstance(v,float) else v for v in row[1:]])  # 输出一个阈值点。
roc_points=[(0.0,0.0)]+[(row[2],row[1]) for row in rows]+[(1.0,1.0)]  # 将 FPR 与 TPR 组织为 ROC 折线点。
pr_points=[(0.0,1.0)]+[(row[1],row[3]) for row in rows]  # 将 recall 与 precision 组织为 PR 折线点。
roc_auc=float(np.trapz([point[1] for point in roc_points],[point[0] for point in roc_points]))  # 用梯形法手写计算 ROC-AUC。
pr_auc=float(np.trapz([point[1] for point in pr_points],[point[0] for point in pr_points]))  # 用梯形法手写计算 PR-AUC。
print('手写 ROC-AUC/PR-AUC=',round(roc_auc,3),round(pr_auc,3))  # 输出两个曲线面积数值。

阈值 | TPR | FPR | Precision | 审核量
0.82 0.333 0.0 1.0 1
0.67 0.667 0.0 1.0 2
0.55 1.0 0.0 1.0 3
0.41 1.0 0.143 0.75 4
0.3 1.0 0.286 0.6 5
0.22 1.0 0.429 0.5 6
0.16 1.0 0.571 0.429 7
0.12 1.0 0.714 0.375 8
0.08 1.0 0.857 0.333 9
0.05 1.0 1.0 0.3 10
手写 ROC-AUC/PR-AUC= 1.0 1.0


## 结果解读

ROC 的 FPR 很小仍可能对应很多正常订单；审核容量固定时，直接看 top-K 的 precision 和 recall 更可操作。

In [4]:
top=np.argsort(score)[::-1][:3]  # 按风险分选择预算内前三笔。
precision=float(y[top].mean())  # 计算预算队列精确率。
recall=float(y[top].sum()/y.sum())  # 计算预算队列召回率。
print('Top3订单=',order[top].tolist())  # 输出人工审核队列。
print('Top3 precision/recall=',round(precision,3),round(recall,3))  # 输出与预算关联的结果。
print('生产差距：需按时间回放、金额成本切片、标签延迟和人工容量监控。')  # 说明生产边界。

Top3订单= ['F03', 'F06', 'F09']
Top3 precision/recall= 1.0 1.0
生产差距：需按时间回放、金额成本切片、标签延迟和人工容量监控。


## 失败案例与修复

只报告 0.7 accuracy 会掩盖全预测正常的无用模型；修复是报告 PR、recall@预算和混淆矩阵。

In [5]:
bad_pred=np.zeros(len(y),dtype=int)  # 构造全预测正常的失败模型。
bad_accuracy=float(np.mean(bad_pred==y))  # 计算其表面准确率。
bad_recall=0.0  # 全预测正常没有欺诈召回。
print('失败 accuracy/recall=',bad_accuracy,bad_recall)  # 输出误导指标。
print('修复 Top3 precision/recall=',precision,recall)  # 输出预算相关指标。
print('AUC 仅衡量排序，不能决定实际阈值。')  # 说明指标边界。

失败 accuracy/recall= 0.7 0.0
修复 Top3 precision/recall= 1.0 1.0
AUC 仅衡量排序，不能决定实际阈值。


In [6]:
assert len(order)>=5  # 保护样本数。
assert precision>prevalence  # 保护模型在 topK 优于随机先验。
assert recall>0  # 保护预算队列召回至少一笔欺诈。
assert bad_accuracy>0.5 and bad_recall==0  # 保护不平衡反例。